# Ask My PDF: Reproducible Project Demo

Educational final project for the *Developing Generative AI Solutions* workshop (SDAIA Academy).

This notebook **imports the real project modules** (`pdf_processor`, `retriever`, `gemini_service`) and runs the pipeline end to end on a **synthetic** PDF.

> **Execution mode:** every answer below is produced with `offline=True`, i.e. the deterministic **offline simulation**, not a live Gemini model. No API key or network access is used. Retrieval (TF-IDF) and the `get_document_stats` tool are real Python code in both modes.

## 1. Setup: locate the project root and import modules

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "app.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from pdf_processor import extract_pages_from_pdf, chunk_pages, compute_document_metrics
from retriever import TFIDFRetriever
from gemini_service import register_document, clear_document_registry, get_document_stats, answer_question
sys.path.insert(0, str(ROOT / "data"))
from generate_sample_pdf import build_sample_pdf

print("Project root:", ROOT)

## 2. Generate the synthetic PDF (3 pages, fictional content)

In [ ]:
pdf_path = build_sample_pdf()
print("Generated:", pdf_path.name)

## 3. Extraction (plain PyMuPDF, no AI) and chunking

In [ ]:
pages = extract_pages_from_pdf(str(pdf_path))
for p in pages:
    print(f"Page {p['page_number']}: {p['text'][:90]}...")

chunks = chunk_pages(pages, chunk_size=150, chunk_overlap=30)
metrics = compute_document_metrics(pages)
print()
print("Chunks:", [c["chunk_id"] for c in chunks])
print("Metrics:", metrics)

## 4. Register the document and build the TF-IDF index

In [ ]:
DOC_ID = "project_aurora_sample.pdf"
clear_document_registry()
register_document(DOC_ID, metrics["page_count"], metrics["word_count"])
retriever = TFIDFRetriever(chunks)

## 5. Real tool call: `get_document_stats` with valid and invalid arguments

In [ ]:
print("Valid:", get_document_stats(DOC_ID))
for bad in ["", "   ", 12345, None]:
    try:
        get_document_stats(bad)
    except (TypeError, ValueError) as err:
        print(f"Invalid {bad!r:>8} -> {type(err).__name__}: {err}")

## 6. Run the synthetic question set (offline simulation)

In [ ]:
questions = json.loads((ROOT / "data" / "sample_questions.json").read_text(encoding="utf-8"))
rows = []
for q in questions:
    top = retriever.retrieve(q["question"], top_k=3)
    answer = answer_question(q["question"], top, DOC_ID, offline=True)
    rows.append({
        "id": q["id"],
        "category": q["category"],
        "top_page": top[0]["page_number"] if top else None,
        "top_score": top[0]["score"] if top else None,
        "mode": top[0].get("retrieval_mode") if top else None,
        "answer": " ".join(answer.split())[:110],
    })

for r in rows:
    print(f"{r['id']:<3} {r['category']:<24} page={r['top_page']} score={r['top_score']:<6} {r['mode']:<7} | {r['answer']}")

## 7. Inspect the evidence behind one answer

Retrieved passages and scores are what make answers traceable. The `PWNED_BY_INJECTION` line on Page 3 is retrieved as **data** but not obeyed.

In [ ]:
question = "What security compliance protocol is enforced?"
for c in retriever.retrieve(question, top_k=3):
    print(f"Page {c['page_number']} | {c['chunk_id']} | score {c['score']:.4f}")
print()
print("Answer:", answer_question(question, retriever.retrieve(question, top_k=3), DOC_ID, offline=True))

## 8. Takeaways

- **Plain extraction:** page text and page/word counts come from PyMuPDF with no model involved.
- **Real retrieval:** TF-IDF cosine similarity with traceable page numbers and scores.
- **Real tool:** `get_document_stats` validates its input and returns exact counts.
- **Simulated generation:** the offline answer engine is rule-based and tuned to this synthetic document; it is a test double, not a language model.
- **Known limitation (Q9):** Arabic questions outside the small keyword lexicon get 0.0 similarity and are refused even when the answer exists.

To try live Gemini generation, use the Streamlit app with a valid `GEMINI_API_KEY` in `.env` (see README).